<a href="https://colab.research.google.com/github/Py-saqlain/Movie_Recap/blob/main/MovieApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Connect to your Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Check the GPU
print("\n--- GPU CHECK ---")
!nvidia-smi -L

# 3. Check for your Movie files
import os
folder_path = '/content/drive/MyDrive/MovieApp'
print("\n--- FILE CHECK ---")
try:
    files = os.listdir(folder_path)
    print(f"SUCCESS! The supercomputer sees your folder! Files inside: {files}")
except FileNotFoundError:
    print(f"ERROR: Still can't find {folder_path}.")

Mounted at /content/drive

--- GPU CHECK ---
GPU 0: Tesla T4 (UUID: GPU-a85dfdbc-31f2-cca8-3e31-d11e277cbe47)

--- FILE CHECK ---
SUCCESS! The supercomputer sees your folder! Files inside: ['1920 Evil Returns 2012 Bluray 1080p.srt', '(2012) 1920_ Evil Returns .mp4', 'movie_index.faiss', 'music', 'recap_timeline.json']


In [2]:
# Install zstd dependency for Ollama
!apt-get install zstd -y

# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# 2. Start the engine in the background
print("\nStarting Ollama server...")
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# 3. Download the Llama 3.2 model
print("Downloading Llama 3.2 (Watch how fast this is)...")
!ollama pull llama3.2
print("\n✅ BRAIN SUCCESSFULLY INSTALLED AND RUNNING! DAY 1 IS COMPLETE!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (569 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current us

In [3]:
## DAY_02
!pip install -q sentence-transformers faiss-cpu opencv-python
print("✅ Vision tools installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.3 MB/s eta 0:00:00
✅ Vision tools installed!


In [4]:
# It asks you for the file names, cleans the text, writes the script, finds the timestamps (RAG), and saves the blueprint.

In [5]:
import os
import re
import requests
import json
import faiss
from sentence_transformers import SentenceTransformer

# --- 1. PROFESSIONAL SETUP (Dynamic Inputs) ---
print("🎬 MOVIE RECAP SETUP")
# Ask user for files (NO HARDCODING)
movie_filename = input("Enter your MOVIE filename (e.g., matrix.mp4): ").strip()
srt_filename = input("Enter your SUBTITLE filename (e.g., matrix.srt): ").strip()
genre = input("Enter the GENRE (horror, action, etc.): ").strip().lower()

# Paths
srt_path = f'/content/drive/MyDrive/MovieApp/{srt_filename}'
index_path = '/content/drive/MyDrive/MovieApp/movie_index.faiss'
timeline_path = '/content/drive/MyDrive/MovieApp/recap_timeline.json'

# --- 2. CLEAN SUBTITLES ---
def clean_subs(path):
    if not os.path.exists(path): return None
    with open(path, 'rb') as f: text = f.read().decode('utf-8', errors='ignore')
    text = re.sub(r'\d{2}:\d{2}:\d{2},\d{3} --> \d{2}:\d{2}:\d{2},\d{3}', '', text) # Time
    text = re.sub(r'^\d+\s*$', '', text, flags=re.MULTILINE) # Index nums
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\!\?]', '', text) # Weird chars
    return " ".join(text.split())

print("\n🧹 Cleaning Subtitles...")
movie_story = clean_subs(srt_path)

if not movie_story:
    print(f"🛑 ERROR: Could not find '{srt_filename}'. Check Drive!")
else:
    # --- 3. GENERATE SCRIPT (Llama 3.2) ---
    print(f"🧠 Generating {genre.upper()} Script (No garbage words)...")
    prompt = f"""
    [SYSTEM: YOU ARE A PROFESSIONAL MOVIE RECAPPER.]
    TASK: Write a 15-minute {genre} recap.
    DATA: {movie_story[:9000]}

    RULES:
    1. Start IMMEDIATELY with "The story begins..."
    2. NO intro/outro/summary.
    3. NO garbage words or AI apologies.
    """

    payload = {"model": "llama3.2", "prompt": prompt, "stream": False}
    try:
        response = requests.post("http://localhost:11434/api/generate", json=payload)
        final_script = response.json()['response']
        print("✅ Script Written Successfully!")
    except:
        print("🛑 ERROR: Llama not running. Did you run Cell 2?")

    # --- 4. VIDEO RAG SEARCH (The "Eyes") ---
    print("\n🔍 Running RAG Search to find timestamps...")
    if os.path.exists(index_path):
        # Load the Vision Engine
        search_model = SentenceTransformer('clip-ViT-B-32')
        index = faiss.read_index(index_path)

        # Split script and search
        sentences = [s.strip() for s in re.split(r'(?<=[.!?]) +', final_script) if len(s.strip()) > 10]
        recap_timeline = []

        for sentence in sentences:
            emb = search_model.encode([sentence])
            faiss.normalize_L2(emb)
            _, indices = index.search(emb, k=1)
            recap_timeline.append({
                "text": sentence,
                "timestamp": int(indices[0][0])
            })

        # --- 5. SAVE BLUEPRINT ---
        with open(timeline_path, 'w') as f:
            json.dump(recap_timeline, f, indent=4)
        print(f"🏆 BLUEPRINT SAVED to {timeline_path}")
        print(f"👉 Ready for Day 4 Audio Sync!")
    else:
        print("🛑 ERROR: 'movie_index.faiss' not found. Did you run Day 2?")

🎬 MOVIE RECAP SETUP
Enter your MOVIE filename (e.g., matrix.mp4): Evil Returns.mp4
Enter your SUBTITLE filename (e.g., matrix.srt): Evil Returns.srt
Enter the GENRE (horror, action, etc.): horror

🧹 Cleaning Subtitles...
🧠 Generating HORROR Script (No garbage words)...
✅ Script Written Successfully!

🔍 Running RAG Search to find timestamps...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


🏆 BLUEPRINT SAVED to /content/drive/MyDrive/MovieApp/recap_timeline.json
👉 Ready for Day 4 Audio Sync!


In [7]:
# edge tts environment
# --- DAY 4: PROFESSIONAL SYNC TOOLS ---
print("🛠️ Installing Edge-TTS (Voice) and MoviePy (Editor)...")
!pip install -q edge-tts moviepy

print("✅ Day 4 Environment Ready.")
print("Waiting for your command to generate Audio...")

🛠️ Installing Edge-TTS (Voice) and MoviePy (Editor)...
✅ Day 4 Environment Ready.
Waiting for your command to generate Audio...


In [9]:
import json
import os
import asyncio
import edge_tts
from moviepy.editor import VideoFileClip, AudioFileClip

# --- 1. SETUP PATHS ---
timeline_path = '/content/drive/MyDrive/MovieApp/recap_timeline.json'
# We need the Movie Path again (re-entering or hardcoding just for this session is fine)
# Since you just entered it, let's assume the variable 'movie_path' is still alive.
# If not, un-comment the line below and put your filename:
movie_path = '/content/drive/MyDrive/MovieApp/Evil Returns.mp4'

output_folder = '/content/drive/MyDrive/MovieApp/Final_Clips'
os.makedirs(output_folder, exist_ok=True)

# --- 2. THE SYNC ENGINE ---
async def create_synced_clips():
    # Load Blueprint
    with open(timeline_path, 'r') as f:
        timeline = json.load(f)

    print(f"🎬 STARTING SYNC ENGINE for {len(timeline)} scenes...")

    # Load the BIG Movie File (This takes a second)
    print("⏳ Loading full movie file... (Please wait)")
    full_movie = VideoFileClip(movie_path)

    for i, item in enumerate(timeline):
        try:
            text = item['text']
            start_time = item['timestamp']

            # A. Generate Audio
            audio_filename = f"voice_{i:03d}.mp3"
            audio_path = os.path.join(output_folder, audio_filename)

            # Using a deep storytelling voice
            communicate = edge_tts.Communicate(text, "en-US-ChristopherNeural")
            await communicate.save(audio_path)

            # B. Measure Duration
            audio_clip = AudioFileClip(audio_path)
            duration = audio_clip.duration

            # C. Cut the Video (The Magic Moment)
            # "Start at the RAG timestamp, and end exactly when the voice ends"
            video_clip = full_movie.subclip(start_time, start_time + duration)

            # D. Combine Audio & Video
            final_clip = video_clip.set_audio(audio_clip)

            # E. Save the Mini-Clip
            output_filename = os.path.join(output_folder, f"scene_{i:03d}.mp4")
            final_clip.write_videofile(output_filename, codec="libx264", audio_codec="aac", logger=None)

            print(f"✅ Scene {i+1}/{len(timeline)} Synced! [Time: {start_time}s | Dur: {duration:.2f}s]")

        except Exception as e:
            print(f"⚠️ Error on Scene {i}: {e}")

    print("\n🎉 DAY 4 COMPLETE! All synced clips are in your Drive folder: 'Final_Clips'")
    full_movie.close()

# --- 3. RUN IT ---
# This line runs the async function in Jupyter/Colab
await create_synced_clips()

🎬 STARTING SYNC ENGINE for 26 scenes...
⏳ Loading full movie file... (Please wait)
✅ Scene 1/26 Synced! [Time: 4633s | Dur: 6.43s]
✅ Scene 2/26 Synced! [Time: 4633s | Dur: 7.99s]
✅ Scene 3/26 Synced! [Time: 4864s | Dur: 4.99s]
✅ Scene 4/26 Synced! [Time: 4583s | Dur: 6.74s]
✅ Scene 5/26 Synced! [Time: 5433s | Dur: 7.51s]
✅ Scene 6/26 Synced! [Time: 5511s | Dur: 10.85s]
✅ Scene 7/26 Synced! [Time: 4580s | Dur: 5.98s]
✅ Scene 8/26 Synced! [Time: 4993s | Dur: 6.26s]
✅ Scene 9/26 Synced! [Time: 289s | Dur: 10.90s]
✅ Scene 10/26 Synced! [Time: 2239s | Dur: 5.04s]
✅ Scene 11/26 Synced! [Time: 5295s | Dur: 4.78s]
✅ Scene 12/26 Synced! [Time: 891s | Dur: 9.34s]
✅ Scene 13/26 Synced! [Time: 5509s | Dur: 7.49s]
✅ Scene 14/26 Synced! [Time: 5509s | Dur: 10.66s]
✅ Scene 15/26 Synced! [Time: 1702s | Dur: 4.87s]
✅ Scene 16/26 Synced! [Time: 1093s | Dur: 4.92s]
✅ Scene 17/26 Synced! [Time: 5511s | Dur: 4.68s]
✅ Scene 18/26 Synced! [Time: 5511s | Dur: 13.85s]
✅ Scene 19/26 Synced! [Time: 1227s | Dur: 

In [ ]:
# cell 09
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np
import re
import os

print("🔍 Waking up the RAG Search Engine (CLIP)...")
# Load the Vision Translator
search_model = SentenceTransformer('clip-ViT-B-32')

# Load your visual vault
db_path = '/content/drive/MyDrive/MovieApp/movie_index.faiss'
if not os.path.exists(db_path):
    print(f"🛑 ERROR: Cannot find {db_path}. Did you mount your drive?")
else:
    index = faiss.read_index(db_path)

    # 1. Split Llama's script into individual sentences
    # This regex splits by periods, exclamation marks, or question marks
    sentences = [s.strip() for s in re.split(r'(?<=[.!?]) +', final_script) if len(s.strip()) > 10]

    print(f"✅ Script split into {len(sentences)} distinct scenes. Starting the visual hunt...\n")

    # 2. The Timeline Vault
    # This list will hold the exact blueprint for tomorrow's audio/video render
    recap_timeline = []

    for sentence in sentences:
        # A. Turn the text into math
        text_embedding = search_model.encode([sentence])
        faiss.normalize_L2(text_embedding)

        # B. Search the FAISS vault for the #1 closest matching frame
        distances, indices = index.search(text_embedding, k=1)

        # C. The matching index IS the timestamp in seconds!
        best_second = int(indices[0][0])

        # D. Save it to our timeline
        recap_timeline.append({
            "text": sentence,
            "timestamp": best_second
        })

    print("🎉 VIDEO RAG COMPLETE! Here is a preview of your sync timeline:\n")
    print("-" * 60)

    # Let's print the first 5 mappings so you can see it working
    for item in recap_timeline[:5]:
        mins = item['timestamp'] // 60
        secs = item['timestamp'] % 60
        print(f"⏱️ Video jumps to [{mins:02d}:{secs:02d}] -> AI Voice says: \"{item['text']}\"")

    print("-" * 60)
    print(f"(...and {len(recap_timeline) - 5} more lines successfully mapped to timestamps!)")


🔍 Waking up the RAG Search Engine (CLIP)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


NameError: name 'final_script' is not defined

In [ ]:
import json

# Define the save path in your Drive
timeline_path = '/content/drive/MyDrive/MovieApp/recap_timeline.json'

# Save the timeline dictionary as a JSON file
with open(timeline_path, 'w') as f:
    json.dump(recap_timeline, f, indent=4)

print(f"✅ TIMELINE BLUEPRINT SAVED SECURELY TO DRIVE!")
print(f"Location: {timeline_path}")
print("-" * 30)
print("DAY 3 cell 10 IS OFFICIALLY COMPLETE. 🏆")

NameError: name 'recap_timeline' is not defined

In [ ]:
# day 04